<div style="background-color: steelblue; padding: 10px; border-radius: 5px;">
    <p style="margin: 10;"></p>
    <h1 style="text-align: center; margin: 0; font-weight: bold; color: white;">SWOT OMIP : Global estimation of spectrum</h1>
    <p style="margin: 10;"></p>
</div>



## 📦**Imports** 

In [24]:
%%time

##################################
#Imports

from datetime import datetime
import s3fs
import xarray as xr
import pyinterp
import numpy as np
import json
from watermark import watermark
import time
import platform
from shapely.geometry import shape, box
from shapely import geometry
import geopandas as gpd
from shapely import geometry
import DataPreprocessing_list as dp

from widetrax import Spectra as sp
import geopandas as gpd

CPU times: user 44 μs, sys: 1 μs, total: 45 μs
Wall time: 48.2 μs


## 🧮 **Required variables**

In [34]:

season="JFM"
prefix='GLO36V1'
s3_folder =f"s3://project-moi-swot-omip/{prefix}" # Do not write /!!!!!!
endpoint_url =  "https://minio.dive.edito.eu/"




## 🔍 **Check the S3 Endpoint**  

In [18]:

fs = s3fs.S3FileSystem(anon=True,endpoint_url=endpoint_url)
# List contents of the bucket
contents = fs.ls(s3_folder)
print("Bucket contents:")
for item in contents:
    print(item)

Bucket contents:
project-moi-swot-omip/GLO36V1/cycle_.keep
project-moi-swot-omip/GLO36V1/cycle_008
project-moi-swot-omip/GLO36V1/cycle_009
project-moi-swot-omip/GLO36V1/cycle_010
project-moi-swot-omip/GLO36V1/cycle_011
project-moi-swot-omip/GLO36V1/cycle_012
project-moi-swot-omip/GLO36V1/cycle_013
project-moi-swot-omip/GLO36V1/cycle_014
project-moi-swot-omip/GLO36V1/cycle_015
project-moi-swot-omip/GLO36V1/cycle_016
project-moi-swot-omip/GLO36V1/cycle_017
project-moi-swot-omip/GLO36V1/cycle_018
project-moi-swot-omip/GLO36V1/cycle_019
project-moi-swot-omip/GLO36V1/cycle_020
project-moi-swot-omip/GLO36V1/cycle_021
project-moi-swot-omip/GLO36V1/cycle_022
project-moi-swot-omip/GLO36V1/cycle_023
project-moi-swot-omip/GLO36V1/cycle_024
project-moi-swot-omip/GLO36V1/cycle_025
project-moi-swot-omip/GLO36V1/cycle_026


## 🔄 **Identify the cycle numbers within the specified time range** 

In [19]:
# Nothing new here

if season=="JFM":    
    start_date = "01012024" # "DDMMYYYY"
    end_date ="31032024"
elif season=="JAS":
    start_date = "01072024" # "DDMMYYYY"
    end_date ="30092024"

if season =="JFM":
    file_path = "https://minio.lab.dive.edito.eu/project-meom-ige/cycles_periods.csv" # works only for winter period
elif season =="JAS":
    file_path = "time_ranges.csv"  # for summer

matching_cycles = dp.get_matching_cycles(file_path, start_date, end_date)

def formater_numeros_concis(liste_numeros):
  return [str(numero).zfill(3) for numero in liste_numeros]
    
matching_cycles = formater_numeros_concis(matching_cycles)
matching_cycles

['008', '009', '010', '011', '012', '013']

In [36]:
%%time
import save_spect_json2 as ssj

import importlib

importlib.reload(dp)

with open("mostly_ocean_boxes_filtered.geojson") as f:
    data = json.load(f)
    
bx=0
for feature in data["features"]:
    poly = shape(feature["geometry"])
    #lon_min, lat_min, lon_max, lat_max
    area = list(poly.bounds)
    area[0]=(area[0] + 360) % 360
    area[2]=(area[2] + 360) % 360
    area = tuple(area)
    print(area)
    lon_min, lat_min, lon_max, lat_max = area

    phase = 'science'
    GEOMETRIES_FILE = f'KaRIn_2kms_{phase}_geometries.geojson'
    pass_numbers = dp.swath_search(lon_min, lon_max, lat_min, lat_max, GEOMETRIES_FILE)
    list_files=dp.find_listdata(matching_cycles, endpoint_url,bucket_name, pass_numbers, start_date, end_date)

    datasets_dict = dp.read_swot_ncfiles_S3folder_list(s3_folder, endpoint_url, area, list_files,engine="h5netcdf")

    
    has_converged, filled_datasets = dp.fill_nan(datasets_dict, varname = "ssh")

    segments_dict = sp.retrieve_segments(filled_datasets,FileType = "NetCDF",namevar="ssh")

    psd_dict, freqs_dict = sp.calculate_psd(segments_dict)
    # Calculate PSD Mean
    psd_mean, freqs_mean = sp.psd_mean_and_freq(psd_dict,freqs_dict)

    ssj.save_spect(("Global_box_"+str(bx)),"GLORYS36",season,freqs_mean,psd_mean,area[0],area[2],area[1],area[3],start_date,end_date)
    bx=bx+1
# I should save for every region or just one file
# In the bucket I have to create a folder to save  outputs

(180.0, -70.0, 190.0, -60.0)
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_504_20240101T060713_20240101T065840_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_513_20240101T135015_20240101T144141_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_530_20240102T042451_20240102T051617_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_532_20240102T060744_20240102T065911_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_541_20240102T135046_20240102T144212_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_558_20240103T042522_20240103T051648_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_569_20240103T135117_20240103T144243_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_009/SWOT_GRID_L3_LR_SSH_009_002_20240104T042553_20240104T051719_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_009/SWOT_GRID_L3_LR_SSH_009_013_20240104T135148_2024010

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(190.0, -70.0, 200.0, -60.0)
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_502_20240101T042420_20240101T051546_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_513_20240101T135015_20240101T144141_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_530_20240102T042451_20240102T051617_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_541_20240102T135046_20240102T144212_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_558_20240103T042522_20240103T051648_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_567_20240103T120823_20240103T125950_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_569_20240103T135117_20240103T144243_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_009/SWOT_GRID_L3_LR_SSH_009_002_20240104T042553_20240104T051719_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_009/SWOT_GRID_L3_LR_SSH_009_011_20240

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(200.0, -70.0, 210.0, -60.0)
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_502_20240101T042420_20240101T051546_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_511_20240101T120721_20240101T125848_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_530_20240102T042451_20240102T051617_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_539_20240102T120752_20240102T125919_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_558_20240103T042522_20240103T051648_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_567_20240103T120823_20240103T125950_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_584_20240104T024259_20240104T033426_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_009/SWOT_GRID_L3_LR_SSH_009_002_20240104T042553_20240104T051719_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_009/SWOT_GRID_L3_LR_SSH_009_011_20240

/opt/anaconda3/envs/s3env/lib/python3.14/site-packages/widetrax/Spectra.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  for col in range(dataset.dims['num_pixels']):


JSON file created
(210.0, -70.0, 220.0, -60.0)
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_500_20240101T024126_20240101T033253_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_511_20240101T120721_20240101T125848_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_528_20240102T024157_20240102T033324_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_539_20240102T120752_20240102T125919_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_556_20240103T024228_20240103T033355_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_567_20240103T120823_20240103T125950_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_008/SWOT_GRID_L3_LR_SSH_008_584_20240104T024259_20240104T033426_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_009/SWOT_GRID_L3_LR_SSH_009_011_20240104T120855_20240104T130021_v1.0.2.nc
project-moi-swot-omip/GLO36V1/cycle_009/SWOT_GRID_L3_LR_SSH_009_028_20240

KeyboardInterrupt: 

## 💾 **Save Results and Information in JSON File**

## 📤 **Export Results to the S3 Endpoint** 

In [13]:
fs = s3fs.S3FileSystem( anon=True, endpoint_url="https://minio.lab.dive.edito.eu", use_ssl=False ) 

In [14]:
json_file = str(nom_region)+"_"+str(name_season)+"_"+str(model)+".json"
fs.put(json_file, "project-meom-ige/OMIP/")

[None]

In [16]:
fs